In [31]:
# =====================================================================================
# セル1: ライブラリの準備とデータ読み込み (修正版)
# =====================================================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
import io
import re
import tkinter as tk
from tkinter import filedialog
import numpy as np

# --- ファイルを読み込んでカラム名を変更するための関数 ---
def load_and_rename_columns(file_path: str) -> pd.DataFrame:
    """
    CSVファイルを読み込み、指定されたカラム名に変更する関数。
    """
    NEW_COLUMN_NAMES = [
        '元ファイル', 'キャビ厚', '射出率(cm³/s)',
        'CH02_立ち上がり時間(s)', 'CH02_立ち上がり圧力(MPa)',
        'CH03_立ち上がり時間(s)', 'CH03_立ち上がり圧力(MPa)',
        'CH04_立ち上がり時間(s)', 'CH04_立ち上がり圧力(MPa)',
        'CH03圧力@CH04立ち上がり時(MPa)', 'CH02圧力@CH03立ち上がり時(MPa)', 'CH02圧力@CH04立ち上がり時(MPa)'
    ]
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='shift_jis')

    if len(df.columns) == len(NEW_COLUMN_NAMES):
        df.columns = NEW_COLUMN_NAMES
    else:
        print(f"警告: {os.path.basename(file_path)} のカラム数が期待値と異なります。カラム名の変更をスキップします。")
        print(f"  > 期待するカラム数: {len(NEW_COLUMN_NAMES)}, 実際のカラム数: {len(df.columns)}")
    return df

# --- ステップ1: ファイルの選択と読み込み ---
print("--- ステップ1: ファイル選択と読み込み ---")
root = tk.Tk()
root.withdraw()
file_paths = filedialog.askopenfilenames(
    title="解析したいCSVファイルを選択してください",
    filetypes=[("CSVファイル", "*.csv"), ("すべてのファイル", "*.*")]
)

if not file_paths:
    print("ファイルが選択されませんでした。処理を中断します。")
else:
    print(f"{len(file_paths)}個のファイルが選択されました。")
    dataframes = {}
    file_path_map = {} # ★修正箇所1: ファイルパスを記憶する辞書を追加
    
    for path in file_paths:
        file_name = os.path.basename(path)
        dataframes[file_name] = load_and_rename_columns(path)
        # ★修正箇所2: 読み込んだファイルの「フォルダパス」を保存
        file_path_map[file_name] = os.path.dirname(path) 
        
    print("すべてのファイルの読み込みと前処理が完了しました。")

--- ステップ1: ファイル選択と読み込み ---
1個のファイルが選択されました。
すべてのファイルの読み込みと前処理が完了しました。


In [ ]:
# =====================================================================================
# セル2: 物理量の計算 (修正済み)
# =====================================================================================

if 'dataframes' in locals() and dataframes:
    print("\n--- ステップ2: データ処理（全計算）---")

    for key, df in dataframes.items():
        print(f"  計算中: {key}")
        # --- 全ての計算を一つのループで実行 ---

        # 1. まず、元の'キャビ厚'列を数値に上書きしてクリーンにする
        df['キャビ厚'] = pd.to_numeric(df['キャビ厚'].astype(str).str.replace('mm', ''), errors='coerce')
        # 2. 次に、セル2内で使われている変数を、クリーンになった列から定義する
        cavity_thickness_numeric = df['キャビ厚']

        distance_L23 = 22.4 #22.4
        distance_L34 = 33   #23
        distance_L24 = distance_L23 + distance_L34
        

        df['時間差ΔT23(s)'] = df['CH03_立ち上がり時間(s)'] - df['CH02_立ち上がり時間(s)']
        df['時間差ΔT34(s)'] = df['CH04_立ち上がり時間(s)'] - df['CH03_立ち上がり時間(s)']
        df['時間差ΔT24(s)'] = df['CH04_立ち上がり時間(s)'] - df['CH02_立ち上がり時間(s)']
        
        df['圧力差ΔP23(Pa)'] = (df['CH02圧力@CH03立ち上がり時(MPa)'] - df['CH03_立ち上がり圧力(MPa)']) * 1000000
        df['圧力差ΔP34(Pa)'] = (df['CH03圧力@CH04立ち上がり時(MPa)'] - df['CH04_立ち上がり圧力(MPa)']) * 1000000
        df['圧力差ΔP24(Pa)'] = (df['CH02圧力@CH04立ち上がり時(MPa)'] - df['CH04_立ち上がり圧力(MPa)']) * 1000000
        
        df['流速V23(mm/s)'] = distance_L23 / df['時間差ΔT23(s)']
        df['流速V34(mm/s)'] = distance_L34 / df['時間差ΔT34(s)']
        df['流速V24(mm/s)'] = distance_L24 / df['時間差ΔT24(s)']
        
        df['流路断面積(mm²)'] = cavity_thickness_numeric * 40
        
        df['流量Q23(mm³/s)'] = df['流速V23(mm/s)'] * df['流路断面積(mm²)']
        df['流量Q34(mm³/s)'] = df['流速V34(mm/s)'] * df['流路断面積(mm²)']
        df['流量Q24(mm³/s)'] = df['流速V24(mm/s)'] * df['流路断面積(mm²)']
        
        df['樹脂粘度η23(Pa・s)'] = (df['圧力差ΔP23(Pa)'] * df['流路断面積(mm²)'] * cavity_thickness_numeric**2) / (12 * distance_L23 * df['流量Q23(mm³/s)'])
        df['せん断速度γ23(1/s)'] = (6 * df['流量Q23(mm³/s)']) / (df['流路断面積(mm²)'] * cavity_thickness_numeric)
        df['樹脂粘度η34(Pa・s)'] = (df['圧力差ΔP34(Pa)'] * df['流路断面積(mm²)'] * cavity_thickness_numeric**2) / (12 * distance_L34 * df['流量Q34(mm³/s)'])
        df['せん断速度γ34(1/s)'] = (6 * df['流量Q34(mm³/s)']) / (df['流路断面積(mm²)'] * cavity_thickness_numeric)
        df['樹脂粘度η24(Pa・s)'] = (df['圧力差ΔP24(Pa)'] * df['流路断面積(mm²)'] * cavity_thickness_numeric**2) / (12 * distance_L24 * df['流量Q24(mm³/s)'])
        df['せん断速度γ24(1/s)'] = (6 * df['流量Q24(mm³/s)']) / (df['流路断面積(mm²)'] * cavity_thickness_numeric)
        
        # --- ▼▼▼ 要求されたコードの追加 (「9行ごとの位置」でグループ化するよう修正) ▼▼▼ ---
        
        # 対象の列
        target_column = 'せん断速度γ23(1/s)'
        
        # 9行ごとのグループIDを作成 (0-8行目がGr.0, 9-17行目がGr.1, ...)
        # (df.indexが0から始まる連番であることを前提)
        group_id_by_position = df.index // 9
        
        # 「位置」に基づいてグループ化
        grouped_shear_rate = df.groupby(group_id_by_position)[target_column]
        
        # 1. 9行ごとの平均値を計算
        mean_col_name = f'{target_column}_射出率ごとの平均'
        df[mean_col_name] = grouped_shear_rate.transform('mean')
        
        # 2. 9行ごとの標準偏差を計算
        std_dev = grouped_shear_rate.transform('std')
        
        # 4. 標準偏差の上限値を計算 (ご要望に基づき: +σ)
        std_upper_col_name = f'{target_column}_標準偏差上'
        df[std_upper_col_name] = std_dev

        # 3. 標準偏差の下限値を計算 (ご要望に基づき: -σ)
        std_lower_col_name = f'{target_column}_標準偏差下'
        df[std_lower_col_name] = -std_dev
        

        # --- ▲▲▲ 追加コードここまで ▲▲▲ ---
        
        dataframes[key] = df
    print("すべての列計算が完了しました。")

else:
    # データがない場合のみエラーメッセージを表示
    print("エラー: ステップ1が実行されていないため、計算するデータがありません。")


--- ステップ2: データ処理（全計算）---
  計算中: FUTABA1_summary_rise_start_results.csv
すべての列計算が完了しました。


In [33]:
# =====================================================================================
# セル3: 集約データのCSV書き出し (保存場所を修正)
# =====================================================================================

print("\n--- ステップ3: 集約データのCSV書き出し ---")

# 'dataframes' と 'file_path_map' がセル1で定義されている前提
if 'dataframes' in locals() and 'file_path_map' in locals():
    
    for key, df in dataframes.items():
        print(f"  集約中: {key}")

        # ★修正箇所1: セル1で記憶した、元のファイルの保存場所を取得
        output_folder = file_path_map[key] 
        
        # 1. グループIDを作成 (セル2と全く同じロジック)
        group_id_by_position = df.index // 9
        
        # 2. 集約したい列のリスト
        columns_to_aggregate = [
            '射出率(cm³/s)', # グループの代表値
            'せん断速度γ23(1/s)_射出率ごとの平均', # グループの平均値
            'せん断速度γ23(1/s)_標準偏差上', # グループの＋σ
            'せん断速度γ23(1/s)_標準偏差下'  # グループの－σ
        ]
        
        # 3. グループ化して、各グループの「最初の行」の値を取得 (9行を1行に集約)
        aggregated_df = df.groupby(group_id_by_position)[columns_to_aggregate].first()
        aggregated_df = aggregated_df.reset_index(drop=True)
        
        # 4. CSVファイルとして保存
        
        # ★修正箇所2: ファイル名を「03_aggregated_summary_(元のファイル名).csv」にする
        # 元のファイル名から拡張子を除去 (例: "data.csv" -> "data")
        base_name = os.path.splitext(key)[0]
        
        # 新しいファイル名を作成 (例: "03_aggregated_summary_data.csv")
        output_filename_leaf = f"03_aggregated_summary_{base_name}.csv"
        
        # ★修正箇所3: 保存場所(フォルダパス)と新しいファイル名を結合
        output_full_path = os.path.join(output_folder, output_filename_leaf)
        
        try:
            aggregated_df.to_csv(output_full_path, index=False, encoding='utf-8-sig')
            # 保存場所をフルパスで表示
            print(f"  完了: {output_full_path} に保存しました。") 
        except Exception as e:
            print(f"  エラー: {key} のCSV保存に失敗しました。 {e}")

else:
    print("エラー: ステップ1が実行されていないため、集約するデータまたはファイルパス情報がありません。")
    print("（セル1を修正して実行し直してください）")


--- ステップ3: 集約データのCSV書き出し ---
  集約中: FUTABA1_summary_rise_start_results.csv
  完了: C:/Users/0uh2j/OneDrive/Desktop/実験データ加工ｐｙ/一回目futabaサンプル9個\FUTABA1_summary_rise_start_results_aggregated_summary.csv に保存しました。


In [34]:
# =====================================================================================
# セル3: 計算結果の途中確認 (修正済み)
# =====================================================================================

if 'dataframes' in locals() and dataframes:
    # コメントを実態に合わせて修正
    print("\n--- ステップ3: 最終データの確認 ---")
    first_df_key = list(dataframes.keys())[0]
    final_df = dataframes[first_df_key]
    print(f"--- 処理結果の最終確認（ファイル: {first_df_key}） ---")

    if len(final_df.columns) > 30:
        display_df = pd.concat([final_df.iloc[:, :5], final_df.iloc[:, -5:]], axis=1)
        print("\n【データの最初の5列と最後の5列】")
        print(display_df.head())
    else:
        # エラーメッセージをより具体的に修正
        print("エラー：計算後の列数が期待値に達していません。セル2の計算が正常に完了したか確認してください。")
        
    for file_name, df_content in dataframes.items():
        print("\n" + "="*50)
        print(f"--- ファイル名: {file_name} ---")
        print("\n【先頭5行】")
        print(df_content.head())
        print("\n【末尾5行】")
        print(df_content.tail())
else:
    print("エラー: データがありません。")


--- ステップ3: 最終データの確認 ---
--- 処理結果の最終確認（ファイル: FUTABA1_summary_rise_start_results.csv） ---

【データの最初の5列と最後の5列】
                元ファイル  キャビ厚  射出率(cm³/s)  CH02_立ち上がり時間(s)  CH02_立ち上がり圧力(MPa)  \
0  002_0.5mm_190℃.csv   0.5        20.4            0.704           0.444977   
1  003_0.5mm_190℃.csv   0.5        20.4            0.704           0.376800   
2  004_0.5mm_190℃.csv   0.5        20.4            0.700           0.339480   
3  005_0.5mm_190℃.csv   0.5        20.4            0.702           0.336681   
4  006_0.5mm_190℃.csv   0.5        20.4            0.706           0.405024   

   樹脂粘度η24(Pa・s)  せん断速度γ24(1/s)  せん断速度γ23(1/s)_射出率ごとの平均  せん断速度γ23(1/s)_標準偏差上  \
0     100.845040    2841.025641             2656.113474            41.530573   
1     102.850438    2805.063291             2656.113474            41.530573   
2     103.212162    2805.063291             2656.113474            41.530573   
3     101.284692    2828.936170             2656.113474            41.530573   
4     102.949528 

In [35]:
#セル3-2
print(final_df.columns) 

Index(['元ファイル', 'キャビ厚', '射出率(cm³/s)', 'CH02_立ち上がり時間(s)', 'CH02_立ち上がり圧力(MPa)',
       'CH03_立ち上がり時間(s)', 'CH03_立ち上がり圧力(MPa)', 'CH04_立ち上がり時間(s)',
       'CH04_立ち上がり圧力(MPa)', 'CH03圧力@CH04立ち上がり時(MPa)', 'CH02圧力@CH03立ち上がり時(MPa)',
       'CH02圧力@CH04立ち上がり時(MPa)', '時間差ΔT23(s)', '時間差ΔT34(s)', '時間差ΔT24(s)',
       '圧力差ΔP23(Pa)', '圧力差ΔP34(Pa)', '圧力差ΔP24(Pa)', '流速V23(mm/s)',
       '流速V34(mm/s)', '流速V24(mm/s)', '流路断面積(mm²)', '流量Q23(mm³/s)',
       '流量Q34(mm³/s)', '流量Q24(mm³/s)', '樹脂粘度η23(Pa・s)', 'せん断速度γ23(1/s)',
       '樹脂粘度η34(Pa・s)', 'せん断速度γ34(1/s)', '樹脂粘度η24(Pa・s)', 'せん断速度γ24(1/s)',
       'せん断速度γ23(1/s)_射出率ごとの平均', 'せん断速度γ23(1/s)_標準偏差上', 'せん断速度γ23(1/s)_標準偏差下'],
      dtype='object')


In [36]:
# =====================================================================================
# セル4: 計算済みデータのCSV保存 (修正済み)
# =====================================================================================

if 'dataframes' in locals() and dataframes:
    print("\n--- ステップ4: 計算結果のCSVファイル保存 ---")
    # output_folderの定義をループの外に移動して効率化
    output_folder = os.path.dirname(file_paths[0])
    
    for original_filename, df_to_save in dataframes.items():
        base_name = os.path.splitext(original_filename)[0]
        new_filename = f"01_processed_{base_name}.csv"
        output_path = os.path.join(output_folder, new_filename)
        try:
            df_to_save.to_csv(output_path, index=False, encoding='utf-8-sig')
            print(f"✅ 保存成功: {output_path}")
        except Exception as e:
            print(f"❌ 保存失敗: {output_path} - {e}")
else:
    print("エラー: 保存するデータがありません。")


--- ステップ4: 計算結果のCSVファイル保存 ---
✅ 保存成功: C:/Users/0uh2j/OneDrive/Desktop/実験データ加工ｐｙ/一回目futabaサンプル9個\FUTABA1_summary_rise_start_results_processed.csv


In [ ]:
# =====================================================================================
# セル5: 【設定】グラフのレイアウトと軸範囲（30行ごと対応版）
# =====================================================================================

# --- ★★★【追加】総集編グラフ①～③用：30行ブロックごとのマーカー形状 ★★★
CV_THICKNESS_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*'] 

# --- ★★★【追加】新しい総集編グラフ（凡例逆転版）用：温度ごとのマーカーとキャビ厚ごとのグラデーション ★★★
TEMP_MARKERS = {'190℃': 'o', '210℃': 's', '230℃': '^'} # 例: 温度ごとのマーカー

# 横に並ぶグラフ間の空白列の数を設定
HORIZONTAL_GRAPH_GAP = 2

# --- グラフの見た目に関する全体設定 ---

# グラフのサイズ（横インチ, 縦インチ）で縦横比を調整
FIGSIZE_NORMAL = (7, 5)        # 通常グラフのサイズ
FIGSIZE_LOGLOG = (8, 7)        # 両対数グラフのサイズ
FIGSIZE_SUMMARY = (8, 6)       # 通常・総合比較グラフのサイズ
FIGSIZE_LOGLOG_SUMMARY = (8, 7)# 両対数・総合比較グラフのサイズ

# プロット（マーカー）のサイズを設定
LINE_MARKER_SIZE = 6      # 折れ線グラフのマーカーサイズ
SCATTER_MARKER_SIZE = 45  # 散布図のマーカーサイズ

# --- 既存の設定 ---

# 横に並ぶグラフ間の空白列の数を設定
HORIZONTAL_GRAPH_GAP = 2

"""""
シートごと、さらに30行ごとのデータブロック（0, 1, 2...）に対して軸範囲と間隔を固定できます。
グラフの軸範囲が被るグラフ群どうしがある場合は、それらのグラフ群のデータ範囲設定行わずに、下記のように、
'default':を 0(データブロックに割り当てられた数字):と合わせて使ってそのグラフ群どうしだけが同じ軸範囲設定になるようにする。
'default': シート全体の基本設定。ここに書かれた設定がまず適用されます。
0, 1, 2...: 特定のデータブロックの設定。'default'の設定を上書きしたい項目だけ記述します。

MANUAL_AXIS_LIMITS = {
    '射出速度と時間差の関係': {
        'default': {'xlim': (0, 220), 'ylim': (0, 0.3), 'xinterval': 20, 'yinterval': 0.05},
         1: {'ylim': (0, 0.4), 'yinterval': 0.1}, # 例：2番目のブロック(31-60行目)だけY軸の範囲を変えたい場合
    },
    '射出速度と流速の関係': {
        'default': {'xlim': (0, 220), 'ylim': (0, 4000), 'xinterval': 20, 'yinterval': 500},
    },
    '射出速度と流量の関係': {
        'default': {'xlim': (0, 220), 'ylim': (0, 200000), 'xinterval': 20, 'yinterval': 50000},
    },
    '射出速度と圧力差の関係': {
        'default': {'xlim': (0, 220), 'ylim': (0, 7 * 10**7), 'xinterval': 20, 'yinterval': 1 * 10**7},
    },
    'せん断速度と樹脂粘度の関係': {
        'default': {'xlim': (10**3, 10**5), 'ylim': (10**0, 10**3), 'xinterval': None, 'yinterval': None},
    }
}
"""""
MANUAL_AXIS_LIMITS = {
    '射出率と時間差の関係': {
              # 1-30行目のキャビ厚0.5mmのグラフ群の設定
             0: {'xlim': (0, 220), 'ylim': (0, 0.25), 'xinterval': 20, 'yinterval': 0.05},
             # 31-60行目のキャビ厚1.0mmのグラフ群の設定
              1: {'xlim': (0, 220), 'ylim': (0, 0.25), 'xinterval': 20, 'yinterval': 0.05},
             # 61-90行目のキャビ厚1.5mmのグラフ群の設定
              2: {'xlim': (0, 220), 'ylim': (0, 0.25), 'xinterval': 20, 'yinterval': 0.05},
             # 91-120行目のキャビ厚2.0mmのグラフ群の設定
              3: {'xlim': (0, 220), 'ylim': (0, 0.25), 'xinterval': 20, 'yinterval': 0.05},
              #総合グラフ用書式
     'all_data': {'xlim': (0, 220), 'ylim': (0, 0.25), 'xinterval': 20, 'yinterval': 0.05},
    },
        


    '射出率と流速の関係': {
        'default': {'xlim': (0, 220), 'ylim': (0, 5000), 'xinterval': 20, 'yinterval': 500},
        'all_data': {'xlim': (0, 220), 'ylim': (0, 5000), 'xinterval': 20, 'yinterval': 500},
    },
    
    # ... 他のシートも同様に設定 ...
    '射出率と流量の関係': {
        'default': {'xlim': (0, 220), 'ylim': (0, 200000), 'xinterval': 20, 'yinterval': 50000},
        'all_data': {'xlim': (0, 220), 'ylim': (0, 200000), 'xinterval': 20, 'yinterval': 50000},
    },

    '射出率と圧力差の関係': {
        'default': {'xlim': (0, 220), 'ylim': (0, 3 * 10**7), 'xinterval': 20, 'yinterval': 0.5 * 10**7},
        'all_data': {'xlim': (0, 220), 'ylim': (0, 3*10**7), 'xinterval': 20, 'yinterval': 0.5*10**7},
    },
    'せん断速度と樹脂粘度の関係': {
        'default': {'xlim': (10**2, 10**5), 'ylim': (10**0, 10**3), 'xinterval': None, 'yinterval': None},
        'all_data': {'xlim': (10**2, 10**5), 'ylim': (10**0, 10**3), 'xinterval': None, 'yinterval': None},
    }
}

print("グラフの全体スタイル設定を読み込みました。")
print(f"グラフ間の空白列を {HORIZONTAL_GRAPH_GAP} 列に設定しました。")
print("グラフの軸範囲・間隔設定（30行ごと）を読み込みました。")

In [ ]:
# =====================================================================================
# セル6: 【設定】総合比較グラフのスタイルを手動で設定
# =====================================================================================

# 各データ系列内の、10行ごとのブロック（0, 1, 2）に対してスタイルを設定します。
# 0: 1-10行目のデータに対応
# 1: 11-20行目のデータに対応
# 2: 21-30行目のデータに対応

SUMMARY_GRAPH_STYLES = {
    # --- 「時間差」シートの総合比較グラフ用スタイル ---
    '時間差ΔT23(s)': {
        0: {'color': None, 'marker': 'o', 'linestyle': '-'},
        1: {'color': None, 'marker': 's', 'linestyle': '--'},
        2: {'color': None, 'marker': '^', 'linestyle': '-.'}
    },
    '時間差ΔT34(s)': {
        0: {'color': None, 'marker': 'o', 'linestyle': '-'},
        1: {'color': None, 'marker': 's', 'linestyle': '--'},
        2: {'color': None, 'marker': '^', 'linestyle': '-.'}
    },
    '時間差ΔT24(s)': {
        0: {'color': None, 'marker': 'o', 'linestyle': '-'},
        1: {'color': None, 'marker': 's', 'linestyle': '--'},
        2: {'color': None, 'marker': '^', 'linestyle': '-.'}
    },

    # --- 「流速」シートの総合比較グラフ用スタイル ---
    '流速V23(mm/s)': {
        0: {'color': None, 'marker': 'o', 'linestyle': '-'},
        1: {'color': None, 'marker': 's', 'linestyle': '--'},
        2: {'color': None, 'marker': '^', 'linestyle': '-.'}
    },
    '流速V34(mm/s)': {
        0: {'color': None, 'marker': 'o', 'linestyle': '-'},
        1: {'color': None, 'marker': 's', 'linestyle': '--'},
        2: {'color': None, 'marker': '^', 'linestyle': '-.'}
    },
    '流速V24(mm/s)': {
        0: {'color': None, 'marker': 'o', 'linestyle': '-'},
        1: {'color': None, 'marker': 's', 'linestyle': '--'},
        2: {'color': None, 'marker': '^', 'linestyle': '-.'}
    },

    # --- 「流量」と「圧力差」も同様に設定 ---
    '流量Q23(mm³/s)': { 0: {'color': None, 'marker': 'o', 'linestyle': '-'}, 1: {'color': None, 'marker': 's', 'linestyle': '--'}, 2: {'color': None, 'marker': '^', 'linestyle': '-.'}},
    '流量Q34(mm³/s)': { 0: {'color': None, 'marker': 'o', 'linestyle': '-'}, 1: {'color': None, 'marker': 's', 'linestyle': '--'}, 2: {'color': None, 'marker': '^', 'linestyle': '-.'}},
    '流量Q24(mm³/s)': { 0: {'color': None, 'marker': 'o', 'linestyle': '-'}, 1: {'color': None, 'marker': 's', 'linestyle': '--'}, 2: {'color': None, 'marker': '^', 'linestyle': '-.'}},
    
    '圧力差ΔP23(Pa)': { 0: {'color': None, 'marker': 'o', 'linestyle': '-'}, 1: {'color': None, 'marker': 's', 'linestyle': '--'}, 2: {'color': None, 'marker': '^', 'linestyle': '-.'}},
    '圧力差ΔP34(Pa)': { 0: {'color': None, 'marker': 'o', 'linestyle': '-'}, 1: {'color': None, 'marker': 's', 'linestyle': '--'}, 2: {'color': None, 'marker': '^', 'linestyle': '-.'}},
    '圧力差ΔP24(Pa)': { 0: {'color': None, 'marker': 'o', 'linestyle': '-'}, 1: {'color': None, 'marker': 's', 'linestyle': '--'}, 2: {'color': None, 'marker': '^', 'linestyle': '-.'}},

    # --- 「樹脂粘度」の総合比較グラフ用スタイル ---
    '樹脂粘度η23(Pa・s)': { 0: {'color': None, 'marker': 'o'}, 1: {'color': None, 'marker': 's'}, 2: {'color': None, 'marker': '^'} },
    '樹脂粘度η34(Pa・s)': { 0: {'color': None, 'marker': 'o'}, 1: {'color': None, 'marker': 's'}, 2: {'color': None, 'marker': '^'} },
    '樹脂粘度η24(Pa・s)': { 0: {'color': None, 'marker': 'o'}, 1: {'color': None, 'marker': 's'}, 2: {'color': None, 'marker': '^'} },
}
print("総合比較グラフのスタイル設定（インデックス別）を読み込みました。")

In [ ]:
# =====================================================================================
# セル7: 【設定】総合比較グラフの色のグラデーションを手動で設定
# =====================================================================================

# データ系列ごとに、グラデーションの開始色と終了色を指定します。
# ここで設定されていないデータ系列は、従来通りの色分け（温度ごとの自動色分け）が適用されます。

GRADIENT_COLOR_SETTINGS = {
    # --- 「時間差」シートの例 ---
    '時間差ΔT23(s)': {'start_color': 'darkred',   'end_color': 'lightpink'},
    '時間差ΔT34(s)': {'start_color': 'darkblue', 'end_color': 'lightblue'},
    '時間差ΔT24(s)': {'start_color': 'darkgreen',  'end_color': 'lightgreen'},

    # --- 「流速」シートの例 ---
    '流速V23(mm/s)': {'start_color': 'darkred',   'end_color': 'lightpink'},
    '流速V34(mm/s)': {'start_color': 'darkblue', 'end_color': 'lightblue'},
    '流速V24(mm/s)': {'start_color': 'darkgreen',  'end_color': 'lightgreen'},

    # --- 「流量」シートの例 ---
    '流量Q23(mm³/s)': {'start_color': 'darkred',   'end_color': 'lightpink'},
    '流量Q34(mm³/s)': {'start_color': 'darkblue', 'end_color': 'lightblue'},
    '流量Q24(mm³/s)': {'start_color': 'darkgreen',  'end_color': 'lightgreen'},
    
    # --- 「圧力差」シートの例 ---
    '圧力差ΔP23(Pa)': {'start_color': 'darkred',   'end_color': 'lightpink'},
    '圧力差ΔP34(Pa)': {'start_color': 'darkblue', 'end_color': 'lightblue'},
    '圧力差ΔP24(Pa)': {'start_color': 'darkgreen',  'end_color': 'lightgreen'},

    # --- 「樹脂粘度」の総合比較グラフ用 ---
    '樹脂粘度η23(Pa・s)': {'start_color': 'darkred',   'end_color': 'lightpink'},
    '樹脂粘度η34(Pa・s)': {'start_color': 'darkblue', 'end_color': 'lightblue'},
    '樹脂粘度η24(Pa・s)': {'start_color': 'darkgreen',  'end_color': 'lightgreen'},
}

# --- 総集編グラフ④用：データ系列ごとの色分け設定 ---
SERIES_COLORS = {
    '時間差ΔT23(s)': 'red', '時間差ΔT34(s)': 'blue', '時間差ΔT24(s)': 'green',
    '流速V23(mm/s)': 'red', '流速V34(mm/s)': 'blue', '流速V24(mm/s)': 'green',
    '流量Q23(mm³/s)': 'red', '流量Q34(mm³/s)': 'blue', '流量Q24(mm³/s)': 'green',
    '圧力差ΔP23(Pa)': 'red', '圧力差ΔP34(Pa)': 'blue', '圧力差ΔP24(Pa)': 'green',
    '樹脂粘度η23(Pa・s)': 'red', '樹脂粘度η34(Pa・s)': 'blue', '樹脂粘度η24(Pa・s)': 'green',
}

# --- ★★★総集編グラフ（凡例逆転版）用：キャビ厚ごとのグラデーション設定 ★★★
CAVITY_THICKNESS_GRADIENT_SETTINGS = {
    0.5: {'start_color': 'red', 'end_color': 'pink'}, 
    1.0: {'start_color': 'blue', 'end_color': 'lightblue'}, 
    1.5: {'start_color': 'gold', 'end_color': 'yellow'}, 
    2.0: {'start_color': 'green', 'end_color': 'lightgreen'}, 
    # 必要に応じてさらに厚みを追加
}

print("総合比較グラフのグラデーション設定を読み込みました。")

In [ ]:
# =====================================================================================
# セル8: グラフ作成ツールの定義（凡例・最終修正版）
# =====================================================================================

print("\n--- グラフ作成の準備 ---")

# --- グラフ全体のスタイル設定 ---
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'

# --- 日本語フォントを自動設定する改善ロジック ---
def set_japanese_font():
    standard_fonts = ['Yu Gothic', 'Meiryo', 'MS Gothic', 'Hiragino Sans', 'IPAexGothic']
    for font in standard_fonts:
        try:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            print(f"日本語フォント '{font}' を自動設定しました。")
            return
        except:
            continue
            
    font_file_path = r"C:\\Users\\0uh2j\\OneDrive\\Desktop\\IPAexfont00401\\ipaexg.ttf"
    if os.path.exists(font_file_path):
        fm.fontManager.addfont(font_file_path)
        font_name = fm.FontProperties(fname=font_file_path).get_name()
        plt.rcParams['font.family'] = font_name
        plt.rcParams['axes.unicode_minus'] = False
        print(f"指定パスの日本語フォント '{font_name}' を設定しました。")
    else:
        print(f"警告: 標準および指定パスに日本語フォントが見つかりません。グラフの日本語が文字化けする可能性があります。")

set_japanese_font()

# --- グラフ番号を管理するためのカウンタークラス ---
class GraphCounter:
    def __init__(self):
        self.count = 1
    def get_next(self):
        num = self.count
        self.count += 1
        return num

# --- ファイル名として安全な文字列を作成するヘルパー関数 ---
def create_safe_filename(prefix, suffix, extension="png"):
    invalid_chars = r'[\\/:*?"<>|]'
    clean_prefix = re.sub(invalid_chars, '_', prefix)
    clean_suffix = re.sub(invalid_chars, '_', suffix)
    return f"{clean_prefix}_{clean_suffix}.{extension}"

# --- グラデーションカラーを生成するヘルパー関数 ---
def generate_color_gradient(start_color, end_color, num_steps):
    from matplotlib.colors import to_rgba
    import numpy as np
    if num_steps <= 1:
        return [to_rgba(start_color)]
    start_rgba = np.array(to_rgba(start_color))
    end_rgba = np.array(to_rgba(end_color))
    gradient = [ tuple(start_rgba + (end_rgba - start_rgba) * i / (num_steps - 1)) for i in range(num_steps) ]
    return gradient

# --- 両対数グラフ用の目盛り設定ヘルパー関数 ---
def set_manual_log_ticks(ax, xlims, ylims):
    if xlims and xlims[0] > 0 and xlims[1] > 0:
        min_exp_x = np.floor(np.log10(xlims[0]))
        max_exp_x = np.ceil(np.log10(xlims[1]))
        ticks_x = [10**i for i in range(int(min_exp_x), int(max_exp_x) + 1)]
        labels_x = [f'$10^{{{int(i)}}}$' if i >= 2 or i <= -2 else f'{10**i:g}' for i in range(int(min_exp_x), int(max_exp_x) + 1)]
        ax.set_xticks(ticks_x)
        ax.set_xticklabels(labels_x)
    if ylims and ylims[0] > 0 and ylims[1] > 0:
        min_exp_y = np.floor(np.log10(ylims[0]))
        max_exp_y = np.ceil(np.log10(ylims[1]))
        ticks_y = [10**i for i in range(int(min_exp_y), int(max_exp_y) + 1)]
        labels_y = [f'$10^{{{int(i)}}}$' if i >= 2 or i <= -2 else f'{10**i:g}' for i in range(int(min_exp_y), int(max_exp_y) + 1)]
        ax.set_yticks(ticks_y)
        ax.set_yticklabels(labels_y)

# --- グラフ作成用の関数1：通常の折れ線グラフ ---
def create_graph_set(df_chunk, workbook, worksheet, x_col, y_col, start_row, color_map, counter, thickness_label, save_folder_path, xlims=None, ylims=None, xinterval=None, yinterval=None):
    print(f"  グラフセットを作成中: X軸={x_col}, Y軸={y_col}")
    graph_title = f'{x_col}と{y_col}の関係'
    graph_row_height_points = 294; graph_col_span = 12; graph_width_pixels_per_col = 65
    worksheet.set_row(start_row, graph_row_height_points)
    fig_all, ax_all = plt.subplots(figsize=FIGSIZE_NORMAL)
    safe_thickness_label = f"CV{str(thickness_label).replace('mm','')}"
    def set_axes_properties(ax, xlims, ylims, xinterval, yinterval):
        if xlims:
            ax.set_xlim(xlims)
            if xinterval:
                ax.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
        if ylims:
            ax.set_ylim(ylims)
            if yinterval:
                ax.set_yticks(np.arange(ylims[0], ylims[1] + yinterval, yinterval))
    segments = range(0, len(df_chunk), 10)
    for idx, i in enumerate(segments):
        current_col_excel = 1 + idx * (graph_col_span + HORIZONTAL_GRAPH_GAP)
        chunk = df_chunk.iloc[i:i+10]
        if chunk.empty: continue
        serial_num = counter.get_next()
        filename = chunk['元ファイル'].iloc[0]; match = re.search(r'(\d+℃)', filename); legend_label = match.group(1) if match else f'Data {i//10 + 1}'
        plot_color = color_map.get(legend_label, 'black')
        fig_ind, ax_ind = plt.subplots(figsize=FIGSIZE_NORMAL)
        ax_ind.plot(chunk[x_col], chunk[y_col], marker='o', linestyle='-', label=legend_label, color=plot_color, markersize=LINE_MARKER_SIZE)
        ax_ind.set_title(f"[{serial_num}] {graph_title}\n(CV厚: {thickness_label} / {legend_label})"); ax_ind.set_xlabel(x_col); ax_ind.set_ylabel(y_col)
        ax_ind.grid(True); ax_ind.legend(loc='best', frameon=False, fontsize='large')
        set_axes_properties(ax_ind, xlims, ylims, xinterval, yinterval)
        fig_ind.tight_layout()
        img_filename = create_safe_filename(f"[{serial_num}]_{safe_thickness_label}_individual", f"{y_col}_{legend_label}")
        fig_ind.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
        img_data_ind = io.BytesIO(); fig_ind.savefig(img_data_ind, format='png'); plt.close(fig_ind)
        worksheet.set_column(current_col_excel, current_col_excel + graph_col_span - 1, graph_width_pixels_per_col / 7)
        worksheet.write(start_row - 1, current_col_excel, f"[{serial_num}] 凡例: {legend_label}")
        worksheet.insert_image(start_row, current_col_excel, '', {'image_data': img_data_ind})
        ax_all.plot(chunk[x_col], chunk[y_col], marker='o', linestyle='-', label=legend_label, color=plot_color, markersize=LINE_MARKER_SIZE)
    serial_num_all = counter.get_next()
    ax_all.set_title(f"[{serial_num_all}] {graph_title}\n(CV厚: {thickness_label} / 統合データ)"); ax_all.set_xlabel(x_col); ax_all.set_ylabel(y_col); ax_all.grid(True)
    ax_all.legend(loc='best', frameon=False, fontsize='large')
    set_axes_properties(ax_all, xlims, ylims, xinterval, yinterval)
    fig_all.tight_layout()
    img_filename_all = create_safe_filename(f"[{serial_num_all}]_{safe_thickness_label}_combined", y_col)
    fig_all.savefig(os.path.join(save_folder_path, img_filename_all), dpi=150)
    num_segments = len(segments)
    current_col_excel_all = 1 + num_segments * (graph_col_span + HORIZONTAL_GRAPH_GAP)
    img_data_all = io.BytesIO(); fig_all.savefig(img_data_all, format='png'); plt.close(fig_all)
    worksheet.set_column(current_col_excel_all, current_col_excel_all + graph_col_span - 1, graph_width_pixels_per_col / 7)
    worksheet.write(start_row - 1, current_col_excel_all, f"[{serial_num_all}] 統合グラフ")
    worksheet.insert_image(start_row, current_col_excel_all, '', {'image_data': img_data_all})

# --- グラフ作成用の関数2：両対数散布図 ---
def create_loglog_scatter_set(df_chunk, workbook, worksheet, x_col, y_col, start_row, color_map, counter, thickness_label, save_folder_path, xlims=None, ylims=None, xinterval=None, yinterval=None):
    print(f"  両対数グラフセットを作成中: X軸={x_col}, Y軸={y_col}")
    graph_title = f'{x_col}と{y_col}の関係'; graph_row_height_points = 375; graph_col_span = 15; graph_width_pixels_per_col = 70
    worksheet.set_row(start_row, graph_row_height_points)
    fig_all, ax_all = plt.subplots(figsize=FIGSIZE_LOGLOG)
    safe_thickness_label = f"CV{str(thickness_label).replace('mm','')}"
    markers = ['o', 's', '^', 'D', 'v', 'p', '*', 'X', 'P', '+']
    
    injection_rate_col_name = None
    possible_names = ['射出率(cm³/s)', '射出速度(mm/s)']
    for name in possible_names:
        if name in df_chunk.columns:
            injection_rate_col_name = name
            break
    
    if injection_rate_col_name:
        injection_speeds = df_chunk[injection_rate_col_name].unique()[:10]
        unit_match = re.search(r'\(.*\)', injection_rate_col_name)
        unit = unit_match.group(0) if unit_match else ''
        marker_labels = [f"{speed} {unit.strip('()')}" for speed in injection_speeds]
    else:
        marker_labels = [f"Rate {i+1}" for i in range(10)]

    marker_handles = [plt.Line2D([0], [0], marker=m, color='black', linestyle='None', markersize=np.sqrt(SCATTER_MARKER_SIZE)) for m in markers]
    
    segments = range(0, len(df_chunk), 10)
    for idx, i in enumerate(segments):
        current_col_excel = 1 + idx * (graph_col_span + HORIZONTAL_GRAPH_GAP)
        chunk = df_chunk.iloc[i:i+10]
        if chunk.empty or chunk[x_col].le(0).any() or chunk[y_col].le(0).any(): continue
        serial_num = counter.get_next()
        filename = chunk['元ファイル'].iloc[0]; match = re.search(r'(\d+℃)', filename); legend_label = match.group(1) if match else f'Data {i//10 + 1}'
        plot_color = color_map.get(legend_label, 'black')
        fig_ind, ax_ind = plt.subplots(figsize=FIGSIZE_LOGLOG)
        for inner_idx, (row_index, row) in enumerate(chunk.iterrows()):
            marker_style = markers[inner_idx % len(markers)]
            ax_ind.scatter(row[x_col], row[y_col], color=plot_color, marker=marker_style, s=SCATTER_MARKER_SIZE)
            if injection_rate_col_name:
                ax_all.scatter(row[x_col], row[y_col], color=plot_color, marker=marker_style, label=f"{legend_label}_{row[injection_rate_col_name]}")
            else:
                ax_all.scatter(row[x_col], row[y_col], color=plot_color, marker=marker_style, label=f"{legend_label}")

        ax_ind.set_xscale('log'); ax_ind.set_yscale('log'); ax_ind.set_title(f"[{serial_num}] {graph_title}\n(CV厚: {thickness_label} / {legend_label})"); ax_ind.set_xlabel(x_col); ax_ind.set_ylabel(y_col)
        ax_ind.grid(False)
        if xlims: ax_ind.set_xlim(xlims)
        if ylims: ax_ind.set_ylim(ylims)
        set_manual_log_ticks(ax_ind, xlims, ylims)

        # ★★★【修正】2つの凡例を作成し、タイトルを右揃えにして重ねて配置 ★★★
        from matplotlib.lines import Line2D
        color_handle = [Line2D([0], [0], marker='o', color=plot_color, linestyle='None')]
        leg1 = ax_ind.legend(color_handle, [legend_label], title='Cylinder temperature', loc='upper right', frameon=False, fontsize='large', title_fontsize='large')
        leg1.get_title().set_ha('right')
        leg2 = ax_ind.legend(marker_handles, marker_labels, title='Injection rate', loc='right', frameon=False, fontsize='large', title_fontsize='large')
        leg2.get_title().set_ha('right')
        ax_ind.add_artist(leg1)

        fig_ind.tight_layout()
        img_filename_ind = create_safe_filename(f"[{serial_num}]_{safe_thickness_label}_individual", f"{y_col}_{legend_label}")
        fig_ind.savefig(os.path.join(save_folder_path, img_filename_ind), dpi=150)
        img_data_ind = io.BytesIO(); fig_ind.savefig(img_data_ind, format='png'); plt.close(fig_ind)
        worksheet.set_column(current_col_excel, current_col_excel + graph_col_span - 1, graph_width_pixels_per_col / 7)
        worksheet.write(start_row - 1, current_col_excel, f"[{serial_num}] 凡例グループ: {legend_label}")
        worksheet.insert_image(start_row, current_col_excel, '', {'image_data': img_data_ind})
        
    serial_num_all = counter.get_next()
    ax_all.set_xscale('log'); ax_all.set_yscale('log'); ax_all.set_title(f"[{serial_num_all}] {graph_title}\n(CV厚: {thickness_label} / 統合データ)"); ax_all.set_xlabel(x_col); ax_all.set_ylabel(y_col)
    ax_all.grid(False)
    if xlims: ax_all.set_xlim(xlims)
    if ylims: ax_all.set_ylim(ylims)
    set_manual_log_ticks(ax_all, xlims, ylims)

    # ★★★【修正】統合グラフの凡例も2つ作成し、それぞれのタイトルを右揃えにする ★★★
    from matplotlib.lines import Line2D
    color_handles_all = [Line2D([0], [0], marker='o', color=c, linestyle='None') for c in color_map.values()]
    color_labels_all = list(color_map.keys())
    leg1_all = ax_all.legend(color_handles_all, color_labels_all, title='Cylinder temperature', loc='upper right', frameon=False, fontsize='large', title_fontsize='large')
    leg1_all.get_title().set_ha('right')
    leg2_all = ax_all.legend(marker_handles, marker_labels, title='Injection rate', loc='right', frameon=False, fontsize='large', title_fontsize='large')
    leg2_all.get_title().set_ha('right')
    ax_all.add_artist(leg1_all)

    fig_all.tight_layout()
    img_filename_all = create_safe_filename(f"[{serial_num_all}]_{safe_thickness_label}_combined", y_col)
    fig_all.savefig(os.path.join(save_folder_path, img_filename_all), dpi=150)
    
    num_segments = len(segments)
    current_col_excel_all = 1 + num_segments * (graph_col_span + HORIZONTAL_GRAPH_GAP)
    img_data_all = io.BytesIO(); fig_all.savefig(img_data_all, format='png'); plt.close(fig_all)
    worksheet.set_column(current_col_excel_all, current_col_excel_all + graph_col_span - 1, graph_width_pixels_per_col / 7)
    worksheet.write(start_row - 1, current_col_excel_all, f"[{serial_num_all}] 統合グラフ")
    worksheet.insert_image(start_row, current_col_excel_all, '', {'image_data': img_data_all})

# --- グラフ作成用の関数3：総合比較グラフ ---
def create_summary_graph(df_chunk, workbook, worksheet, x_col, y_cols, start_row, color_map, y_label, counter, thickness_label, save_folder_path, xlims=None, ylims=None, xinterval=None, yinterval=None):
    print(f"  総合比較グラフを作成中: Y軸={y_label}")
    graph_title = f'{y_label.split(" ")[0]} 総合比較グラフ'
    worksheet.set_row(start_row, 300)
    fig, ax = plt.subplots(figsize=FIGSIZE_SUMMARY)
    safe_thickness_label = f"CV{str(thickness_label).replace('mm','')}"
    color_palettes = {}
    num_segments = len(df_chunk) // 10 if len(df_chunk) >= 10 else 1
    for y_col in y_cols:
        if y_col in GRADIENT_COLOR_SETTINGS:
            settings = GRADIENT_COLOR_SETTINGS[y_col]
            color_palettes[y_col] = generate_color_gradient(settings['start_color'], settings['end_color'], num_segments)
        else:
            color_palettes[y_col] = None
    for i in range(0, len(df_chunk), 10):
        segment_index = i // 10
        chunk = df_chunk.iloc[i:i+10]
        if chunk.empty: continue
        filename = chunk['元ファイル'].iloc[0]
        match = re.search(r'(\d+℃)', filename)
        legend_label_temp = match.group(1) if match else f'Data {segment_index + 1}'
        for y_col in y_cols:
            series_styles = SUMMARY_GRAPH_STYLES.get(y_col, {})
            style_settings = series_styles.get(segment_index, {})
            plot_color = style_settings.get('color')
            if plot_color is None:
                if color_palettes.get(y_col) is not None:
                    plot_color = color_palettes[y_col][segment_index]
                else:
                    plot_color = color_map.get(legend_label_temp, 'black')
            marker_style = style_settings.get('marker', 'o')
            line_style = style_settings.get('linestyle', '-')
            label_suffix_match = re.search(r'([ΔVQPγ]\w+)', y_col)
            label_suffix = label_suffix_match.group(1) if label_suffix_match else y_col
            full_label = f'{legend_label_temp} ({label_suffix})'
            ax.plot(chunk[x_col], chunk[y_col], marker=marker_style, linestyle=line_style, color=plot_color, label=full_label, markersize=LINE_MARKER_SIZE)
    serial_num = counter.get_next()
    ax.set_title(f"[{serial_num}] {graph_title}\n(CV厚: {thickness_label})")
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_label)
    ax.grid(True)
    ax.legend(loc='best', frameon=False, fontsize='large', markerscale=1.5)
    if xlims:
        ax.set_xlim(xlims)
        if xinterval:
            ax.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
    if ylims:
        ax.set_ylim(ylims)
        if yinterval:
            ax.set_yticks(np.arange(ylims[0], ylims[1] + yinterval, yinterval))
    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_{safe_thickness_label}_summary", y_label)
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO()
    fig.savefig(img_data, format='png')
    plt.close(fig)
    worksheet.set_column(1, 1 + 12 - 1, 8)
    worksheet.write(start_row - 1, 1, f"[{serial_num}] {graph_title}")
    worksheet.insert_image(start_row, 1, '', {'image_data': img_data})

# --- グラフ作成用の関数4：両対数・総合比較グラフ ---
def create_loglog_summary_graph(df_chunk, workbook, worksheet, xy_cols, start_row, color_map, counter, thickness_label, save_folder_path, xlims=None, ylims=None):
    print(f"  両対数・総合比較グラフを作成中...")
    graph_title = 'せん断速度と樹脂粘度 総合比較'
    worksheet.set_row(start_row, 375)
    fig, ax = plt.subplots(figsize=FIGSIZE_LOGLOG_SUMMARY)
    safe_thickness_label = f"CV{str(thickness_label).replace('mm','')}"
    num_segments = len(df_chunk) // 10 if len(df_chunk) >= 10 else 1
    color_palettes = {}
    for x_col, y_col in xy_cols:
        if y_col in GRADIENT_COLOR_SETTINGS:
            settings = GRADIENT_COLOR_SETTINGS[y_col]
            color_palettes[y_col] = generate_color_gradient(settings['start_color'], settings['end_color'], num_segments)
        else:
            color_palettes[y_col] = None
    for i in range(0, len(df_chunk), 10):
        segment_index = i // 10
        chunk = df_chunk.iloc[i:i+10]
        if chunk.empty: continue
        filename = chunk['元ファイル'].iloc[0]
        match = re.search(r'(\d+℃)', filename)
        legend_label_temp = match.group(1) if match else f'Data {segment_index + 1}'
        for x_col, y_col in xy_cols:
            if chunk[x_col].le(0).any() or chunk[y_col].le(0).any(): continue
            series_styles = SUMMARY_GRAPH_STYLES.get(y_col, {})
            style_settings = series_styles.get(segment_index, {})
            plot_color = style_settings.get('color')
            if plot_color is None:
                if color_palettes.get(y_col) is not None:
                    plot_color = color_palettes[y_col][segment_index]
                else:
                    plot_color = color_map.get(legend_label_temp, 'black')
            marker_style = style_settings.get('marker', 'o')
            label_suffix_match = re.search(r'([η]\w+)', y_col)
            label_suffix = label_suffix_match.group(1) if label_suffix_match else y_col
            full_label = f'{legend_label_temp} ({label_suffix})'
            ax.scatter(chunk[x_col], chunk[y_col], marker=marker_style, color=plot_color, label=full_label, s=SCATTER_MARKER_SIZE)
    serial_num = counter.get_next()
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_title(f"[{serial_num}] {graph_title}\n(CV厚: {thickness_label})")
    ax.set_xlabel('せん断速度 (1/s)')
    ax.set_ylabel('樹脂粘度 (Pa・s)')
    ax.grid(False)
    if xlims: ax.set_xlim(xlims)
    if ylims: ax.set_ylim(ylims)
    set_manual_log_ticks(ax, xlims, ylims)
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), frameon=False, fontsize='large', markerscale=1.5)
    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_{safe_thickness_label}_summary", "loglog_comparison")
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO()
    fig.savefig(img_data, format='png')
    plt.close(fig)
    worksheet.set_column(1, 1 + 15 - 1, 8)
    worksheet.write(start_row - 1, 1, f"[{serial_num}] {graph_title}")
    worksheet.insert_image(start_row, 1, '', {'image_data': img_data})

print("4種類のグラフ作成用関数を定義し、各種スタイル設定を反映するように修正しました。")

In [ ]:
# =====================================================================================
# セル9: 総集編グラフの作成関数（凡例自動配置・最前面版）
# =====================================================================================

print("\n--- グラフ作成の準備（2/2）: 総集編グラフ用の関数を定義 ---")

# --- 総集編グラフ①～③用 (通常スケール) ---
def create_final_individual_graph(df_full, workbook, worksheet, x_col, y_col, start_row, color_map, counter, save_folder_path, start_col=1, xlims=None, ylims=None, xinterval=None, yinterval=None):
    print(f"  総集編・個別グラフを作成中: {y_col}")
    serial_num = counter.get_next()
    graph_title = f'全データ: {x_col}と{y_col}の関係'
    fig, ax = plt.subplots(figsize=FIGSIZE_NORMAL)
    graph_col_span = 12

    if '温度' not in df_full.columns:
        df_full['温度'] = pd.to_numeric(df_full['元ファイル'].str.extract(r'(\d+)℃')[0], errors='coerce')

    cv_thickness_labels = []
    for i in range(0, len(df_full), 30):
        chunk_30 = df_full.iloc[i:i+30]
        if chunk_30.empty: continue
        chunk_index = i // 30
        marker = CV_THICKNESS_MARKERS[chunk_index % len(CV_THICKNESS_MARKERS)]
        thickness = chunk_30['キャビ厚'].iloc[0]
        if not any(t == thickness for m, t in cv_thickness_labels):
            cv_thickness_labels.append((marker, thickness))
        
        for temp_val in sorted(df_full['温度'].unique()):
            chunk_temp = chunk_30[chunk_30['温度'] == temp_val]
            if chunk_temp.empty: continue
            temp_label = f"{int(temp_val)}℃"
            color = color_map.get(temp_label, 'black')
            ax.plot(chunk_temp[x_col], chunk_temp[y_col], marker=marker, color=color, linestyle='-', markersize=LINE_MARKER_SIZE, label=temp_label)
    
    ax.set_title(f"[{serial_num}] {graph_title}"); ax.set_xlabel(x_col); ax.set_ylabel(y_col); ax.grid(True)
    if xlims: ax.set_xlim(xlims)
    if ylims: ax.set_ylim(ylims)
    if xinterval: ax.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
    if yinterval: ax.set_yticks(np.arange(ylims[0], ylims[1] + yinterval, yinterval))

    # ★★★【修正】凡例を単一化し、自動配置('best')と最前面表示(zorder)を設定 ★★★
    from matplotlib.lines import Line2D
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    
    dummy_handle = Line2D([0], [0], marker='o', color='none', linestyle='none')
    temp_handles = by_label.values()
    temp_labels = by_label.keys()
    marker_handles = [plt.Line2D([0], [0], marker=m, color='gray', linestyle='None') for m, t in cv_thickness_labels]
    marker_labels = [str(t) for m, t in cv_thickness_labels]

    all_handles = [dummy_handle] + list(temp_handles) + [dummy_handle] + marker_handles
    all_labels = ['Temperature'] + list(temp_labels) + ['Cavity Thickness'] + marker_labels
    
    legend = ax.legend(all_handles, all_labels, loc='best', frameon=False, fontsize='large')
    for text in legend.get_texts():
        if text.get_text() in ['Temperature', 'Cavity Thickness']:
            text.set_weight('bold')

    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_final_individual", y_col)
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO(); fig.savefig(img_data, format='png'); plt.close(fig)
    worksheet.set_column(start_col, start_col + graph_col_span - 1, 8)
    worksheet.insert_image(start_row, start_col, '', {'image_data': img_data})
    return graph_col_span

# --- 総集編グラフ④用 (通常スケール) ---
def create_final_summary_graph(df_full, workbook, worksheet, x_col, y_cols, start_row, color_map, counter, save_folder_path, start_col=1, xlims=None, ylims=None, xinterval=None, yinterval=None):
    print(f"  総集編・総合比較グラフを作成中...")
    serial_num = counter.get_next()
    graph_title = '全データ 総合比較'
    fig, ax = plt.subplots(figsize=FIGSIZE_SUMMARY)
    graph_col_span = 12

    cv_thickness_labels = []
    for i in range(0, len(df_full), 30):
        chunk_30 = df_full.iloc[i:i+30]
        if chunk_30.empty: continue
        chunk_index = i // 30
        marker = CV_THICKNESS_MARKERS[chunk_index % len(CV_THICKNESS_MARKERS)]
        thickness = chunk_30['キャビ厚'].iloc[0]
        if not any(t == thickness for m, t in cv_thickness_labels):
            cv_thickness_labels.append((marker, thickness))
        
        for y_col in y_cols:
            color = SERIES_COLORS.get(y_col, 'black')
            linestyle = '-'
            label = y_col if i == 0 else None
            ax.plot(chunk_30[x_col], chunk_30[y_col], marker=marker, color=color, linestyle=linestyle, markersize=LINE_MARKER_SIZE, label=label)

    ax.set_title(f"[{serial_num}] {graph_title}"); ax.set_xlabel(x_col); ax.set_ylabel("Value"); ax.grid(True)
    if xlims: ax.set_xlim(xlims)
    if ylims: ax.set_ylim(ylims)
    if xinterval: ax.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
    if yinterval: ax.set_yticks(np.arange(ylims[0], ylims[1] + yinterval, yinterval))
    
    # ★★★【修正】凡例を単一化し、自動配置('best')と最前面表示(zorder)を設定 ★★★
    from matplotlib.lines import Line2D
    dummy_handle = Line2D([0], [0], marker='o', color='none', linestyle='none')
    unique_labels = sorted(list(set(y_cols)))
    color_handles = [plt.Line2D([0], [0], marker='o', color=SERIES_COLORS.get(l, 'k'), linestyle='-') for l in unique_labels]
    marker_handles = [plt.Line2D([0], [0], marker=m, color='gray', linestyle='None') for m, t in cv_thickness_labels]
    marker_labels = [str(t) for m, t in cv_thickness_labels]
    
    all_handles = [dummy_handle] + color_handles + [dummy_handle] + marker_handles
    all_labels = ['Series'] + unique_labels + ['Cavity Thickness'] + marker_labels
    
    legend = ax.legend(all_handles, all_labels, loc='best', frameon=False, fontsize='large')
    for text in legend.get_texts():
        if text.get_text() in ['Series', 'Cavity Thickness']:
            text.set_weight('bold')

    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_final_summary", "all_series")
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO(); fig.savefig(img_data, format='png'); plt.close(fig)
    worksheet.set_column(start_col, start_col + graph_col_span - 1, 8)
    worksheet.insert_image(start_row, start_col, '', {'image_data': img_data})
    return graph_col_span

# --- 総集編グラフ①～③用 (両対数スケール) ---
def create_final_loglog_individual_graph(df_full, workbook, worksheet, x_col, y_col, start_row, color_map, counter, save_folder_path, start_col=1, xlims=None, ylims=None):
    print(f"  総集編・個別グラフ(LogLog)を作成中: {y_col}")
    serial_num = counter.get_next()
    graph_title = f'全データ: {x_col}と{y_col}の関係'
    fig, ax = plt.subplots(figsize=FIGSIZE_LOGLOG)
    ax.set_xscale('log'); ax.set_yscale('log')
    graph_col_span = 15

    if '温度' not in df_full.columns:
        df_full['温度'] = pd.to_numeric(df_full['元ファイル'].str.extract(r'(\d+)℃')[0], errors='coerce')

    cv_thickness_labels = []
    for i in range(0, len(df_full), 30):
        chunk_30 = df_full.iloc[i:i+30]
        if chunk_30.empty: continue
        chunk_index = i // 30
        marker = CV_THICKNESS_MARKERS[chunk_index % len(CV_THICKNESS_MARKERS)]
        thickness = chunk_30['キャビ厚'].iloc[0]
        if not any(t == thickness for m, t in cv_thickness_labels):
            cv_thickness_labels.append((marker, thickness))
        
        for temp_val in sorted(df_full['温度'].unique()):
            chunk_temp = chunk_30[chunk_30['温度'] == temp_val]
            if chunk_temp.empty or chunk_temp[x_col].le(0).any() or chunk_temp[y_col].le(0).any(): continue
            temp_label = f"{int(temp_val)}℃"
            color = color_map.get(temp_label, 'black')
            ax.scatter(chunk_temp[x_col], chunk_temp[y_col], marker=marker, color=color, s=SCATTER_MARKER_SIZE, label=temp_label)

    ax.set_title(f"[{serial_num}] {graph_title}"); ax.set_xlabel(x_col); ax.set_ylabel(y_col); ax.grid(False)
    if xlims: ax.set_xlim(xlims)
    if ylims: ax.set_ylim(ylims)
    set_manual_log_ticks(ax, xlims, ylims)

    # ★★★【修正】凡例を単一化し、自動配置('best')と最前面表示(zorder)を設定 ★★★
    from matplotlib.lines import Line2D
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    
    dummy_handle = Line2D([0], [0], marker='o', color='none', linestyle='none')
    temp_handles = by_label.values()
    temp_labels = by_label.keys()
    marker_handles = [plt.Line2D([0], [0], marker=m, color='gray', linestyle='None') for m, t in cv_thickness_labels]
    marker_labels = [str(t) for m, t in cv_thickness_labels]

    all_handles = [dummy_handle] + list(temp_handles) + [dummy_handle] + marker_handles
    all_labels = ['Temperature'] + list(temp_labels) + ['Cavity Thickness'] + marker_labels
    
    legend = ax.legend(all_handles, all_labels, loc='best', frameon=False, fontsize='large')
    for text in legend.get_texts():
        if text.get_text() in ['Temperature', 'Cavity Thickness']:
            text.set_weight('bold')

    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_final_individual", y_col)
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO(); fig.savefig(img_data, format='png'); plt.close(fig)
    worksheet.set_column(start_col, start_col + graph_col_span - 1, 8)
    worksheet.insert_image(start_row, start_col, '', {'image_data': img_data})
    return graph_col_span

# --- 総集編グラフ④用 (両対数スケール) ---
def create_final_loglog_summary_graph(df_full, workbook, worksheet, xy_cols, start_row, color_map, counter, save_folder_path, start_col=1, xlims=None, ylims=None):
    print(f"  総集編・総合比較グラフ(LogLog)を作成中...")
    serial_num = counter.get_next()
    graph_title = '全データ 総合比較'
    fig, ax = plt.subplots(figsize=FIGSIZE_LOGLOG_SUMMARY)
    ax.set_xscale('log'); ax.set_yscale('log')
    graph_col_span = 15

    cv_thickness_labels = []
    for i in range(0, len(df_full), 30):
        chunk_30 = df_full.iloc[i:i+30]
        if chunk_30.empty: continue
        chunk_index = i // 30
        marker = CV_THICKNESS_MARKERS[chunk_index % len(CV_THICKNESS_MARKERS)]
        thickness = chunk_30['キャビ厚'].iloc[0]
        if not any(t == thickness for m, t in cv_thickness_labels):
            cv_thickness_labels.append((marker, thickness))
        
        for x_col, y_col in xy_cols:
            if chunk_30[x_col].le(0).any() or chunk_30[y_col].le(0).any(): continue
            color = SERIES_COLORS.get(y_col, 'black')
            ax.scatter(chunk_30[x_col], chunk_30[y_col], marker=marker, color=color, s=SCATTER_MARKER_SIZE, label=y_col)
    
    ax.set_title(f"[{serial_num}] {graph_title}"); ax.set_xlabel("Shear Rate (1/s)"); ax.set_ylabel("Viscosity (Pa·s)"); ax.grid(False)
    if xlims: ax.set_xlim(xlims)
    if ylims: ax.set_ylim(ylims)
    set_manual_log_ticks(ax, xlims, ylims)

    # ★★★【修正】凡例を単一化し、自動配置('best')と最前面表示(zorder)を設定 ★★★
    from matplotlib.lines import Line2D
    dummy_handle = Line2D([0], [0], marker='o', color='none', linestyle='none')
    unique_labels = sorted(list(set(item[1] for item in xy_cols)))
    color_handles = [plt.Line2D([0], [0], marker='o', color=SERIES_COLORS.get(l, 'k'), linestyle='None') for l in unique_labels]
    marker_handles = [plt.Line2D([0], [0], marker=m, color='gray', linestyle='None') for m, t in cv_thickness_labels]
    marker_labels = [str(t) for m, t in cv_thickness_labels]
    
    all_handles = [dummy_handle] + color_handles + [dummy_handle] + marker_handles
    all_labels = ['Series'] + unique_labels + ['Cavity Thickness'] + marker_labels
    
    legend = ax.legend(all_handles, all_labels, loc='best', frameon=False, fontsize='large')
    for text in legend.get_texts():
        if text.get_text() in ['Series', 'Cavity Thickness']:
            text.set_weight('bold')
    
    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_final_summary", "all_series_loglog")
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO(); fig.savefig(img_data, format='png'); plt.close(fig)
    worksheet.set_column(start_col, start_col + graph_col_span - 1, 8)
    worksheet.insert_image(start_row, start_col, '', {'image_data': img_data})
    return graph_col_span

# --- 凡例逆転版・個別総集編グラフ (通常スケール) ---
def create_final_individual_graph_reversed(df_full, workbook, worksheet, x_col, y_col, start_row, counter, save_folder_path, start_col=1, xlims=None, ylims=None, xinterval=None, yinterval=None):
    print(f"  総集編・個別グラフ（凡例逆転）を作成中: {y_col}")
    serial_num = counter.get_next()
    graph_title = f'全データ: {x_col}と{y_col}の関係 (凡例逆転)'
    fig, ax = plt.subplots(figsize=FIGSIZE_NORMAL)
    graph_col_span = 12
    
    if '温度' not in df_full.columns:
        df_full['温度'] = pd.to_numeric(df_full['元ファイル'].str.extract(r'(\d+)℃')[0], errors='coerce')
    unique_thicknesses = sorted(df_full['キャビ厚'].unique())
    unique_temps = sorted(df_full['温度'].unique())
    
    for thickness_val in unique_thicknesses:
        df_thick = df_full[df_full['キャビ厚'] == thickness_val]
        grad_settings = CAVITY_THICKNESS_GRADIENT_SETTINGS.get(thickness_val, {'start_color':'lightgray', 'end_color':'black'})
        color_grad = generate_color_gradient(grad_settings['start_color'], grad_settings['end_color'], len(unique_temps))
        
        for i, temp_val in enumerate(unique_temps):
            df_temp = df_thick[df_thick['温度'] == temp_val]
            if df_temp.empty: continue
            temp_label = f"{int(temp_val)}℃"
            marker = TEMP_MARKERS.get(temp_label, 'x')
            color = color_grad[i % len(color_grad)]
            ax.plot(df_temp[x_col], df_temp[y_col], marker=marker, color=color, linestyle='-')

    ax.set_title(f"[{serial_num}] {graph_title}"); ax.set_xlabel(x_col); ax.set_ylabel(y_col); ax.grid(True)
    if xlims: ax.set_xlim(xlims)
    if ylims: ax.set_ylim(ylims)
    if xinterval: ax.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
    if yinterval: ax.set_yticks(np.arange(ylims[0], ylims[1] + yinterval, yinterval))

    # ★★★【修正】凡例を単一化し、自動配置('best')と最前面表示(zorder)を設定 ★★★
    from matplotlib.lines import Line2D
    dummy_handle = Line2D([0], [0], marker='o', color='none', linestyle='none')
    temp_handles = [plt.Line2D([0], [0], marker=m, color='gray', linestyle='None') for t, m in TEMP_MARKERS.items() if int(t.replace('℃','')) in unique_temps]
    temp_labels = [t for t, m in TEMP_MARKERS.items() if int(t.replace('℃','')) in unique_temps]
    cavity_handles = [plt.Rectangle((0,0), 1, 1, color=generate_color_gradient(s['start_color'], s['end_color'], 3)[1]) for t, s in CAVITY_THICKNESS_GRADIENT_SETTINGS.items() if t in unique_thicknesses]
    cavity_labels = [f"{t}mm" for t in unique_thicknesses]

    all_handles = [dummy_handle] + temp_handles + [dummy_handle] + cavity_handles
    all_labels = ['Temperature'] + temp_labels + ['Cavity Thickness (Gradient)'] + cavity_labels
    
    legend = ax.legend(all_handles, all_labels, loc='best', frameon=False, fontsize='large')
    for text in legend.get_texts():
        if text.get_text() in ['Temperature', 'Cavity Thickness (Gradient)']:
            text.set_weight('bold')

    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_final_individual_rev", y_col)
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO(); fig.savefig(img_data, format='png'); plt.close(fig)
    worksheet.set_column(start_col, start_col + graph_col_span - 1, 8)
    worksheet.insert_image(start_row, start_col, '', {'image_data': img_data})
    return graph_col_span

# --- 凡例逆転版・総集編グラフ (通常スケール) ---
def create_final_summary_graph_reversed(df_full, workbook, worksheet, x_col, y_cols, start_row, counter, save_folder_path, start_col=1, xlims=None, ylims=None, xinterval=None, yinterval=None):
    print(f"  総集編・総合比較グラフ（凡例逆転）を作成中...")
    serial_num = counter.get_next()
    graph_title = '全データ 総合比較（凡例逆転）'
    fig, ax = plt.subplots(figsize=FIGSIZE_SUMMARY)
    graph_col_span = 12
    
    if '温度' not in df_full.columns:
        df_full['温度'] = pd.to_numeric(df_full['元ファイル'].str.extract(r'(\d+)℃')[0], errors='coerce')
    unique_thicknesses = sorted(df_full['キャビ厚'].unique())
    unique_temps = sorted(df_full['温度'].unique())
    
    for thickness_val in unique_thicknesses:
        df_thick = df_full[df_full['キャビ厚'] == thickness_val]
        grad_settings = CAVITY_THICKNESS_GRADIENT_SETTINGS.get(thickness_val, {'start_color':'lightgray', 'end_color':'black'})
        color_grad = generate_color_gradient(grad_settings['start_color'], grad_settings['end_color'], len(unique_temps))
        
        for i, temp_val in enumerate(unique_temps):
            df_temp = df_thick[df_thick['温度'] == temp_val]
            if df_temp.empty: continue
            temp_label = f"{int(temp_val)}℃"
            marker = TEMP_MARKERS.get(temp_label, 'x')
            color = color_grad[i % len(color_grad)]
            for y_col in y_cols:
                ax.plot(df_temp[x_col], df_temp[y_col], marker=marker, color=color, linestyle='-')

    ax.set_title(f"[{serial_num}] {graph_title}"); ax.set_xlabel(x_col); ax.set_ylabel("Value"); ax.grid(True)
    if xlims: ax.set_xlim(xlims);
    if ylims: ax.set_ylim(ylims)
    if xinterval: ax.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
    if yinterval: ax.set_yticks(np.arange(ylims[0], ylims[1] + yinterval, yinterval))

    # ★★★【修正】凡例を単一化し、自動配置('best')と最前面表示(zorder)を設定 ★★★
    from matplotlib.lines import Line2D
    dummy_handle = Line2D([0], [0], marker='o', color='none', linestyle='none')
    temp_handles = [plt.Line2D([0], [0], marker=m, color='gray', linestyle='None') for t, m in TEMP_MARKERS.items() if int(t.replace('℃','')) in unique_temps]
    temp_labels = [t for t, m in TEMP_MARKERS.items() if int(t.replace('℃','')) in unique_temps]
    cavity_handles = [plt.Rectangle((0,0), 1, 1, color=generate_color_gradient(s['start_color'], s['end_color'], 3)[1]) for t, s in CAVITY_THICKNESS_GRADIENT_SETTINGS.items() if t in unique_thicknesses]
    cavity_labels = [f"{t}mm" for t in unique_thicknesses]

    all_handles = [dummy_handle] + temp_handles + [dummy_handle] + cavity_handles
    all_labels = ['Temperature'] + temp_labels + ['Cavity Thickness (Gradient)'] + cavity_labels
    
    legend = ax.legend(all_handles, all_labels, loc='best', frameon=False, fontsize='large')
    for text in legend.get_texts():
        if text.get_text() in ['Temperature', 'Cavity Thickness (Gradient)']:
            text.set_weight('bold')
    
    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_final_summary_rev", "all_series")
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO(); fig.savefig(img_data, format='png'); plt.close(fig)
    worksheet.set_column(start_col, start_col + graph_col_span - 1, 8)
    worksheet.insert_image(start_row, start_col, '', {'image_data': img_data})
    return graph_col_span

# --- 凡例逆転版・個別総集編グラフ (両対数スケール) ---
def create_final_loglog_individual_graph_reversed(df_full, workbook, worksheet, x_col, y_col, start_row, counter, save_folder_path, start_col=1, xlims=None, ylims=None):
    print(f"  総集編・個別グラフ(LogLog, 凡例逆転)を作成中: {y_col}")
    serial_num = counter.get_next()
    graph_title = f'全データ: {x_col}と{y_col}の関係 (凡例逆転)'
    fig, ax = plt.subplots(figsize=FIGSIZE_LOGLOG)
    ax.set_xscale('log'); ax.set_yscale('log')
    graph_col_span = 15

    if '温度' not in df_full.columns:
        df_full['温度'] = pd.to_numeric(df_full['元ファイル'].str.extract(r'(\d+)℃')[0], errors='coerce')
    unique_thicknesses = sorted(df_full['キャビ厚'].unique())
    unique_temps = sorted(df_full['温度'].unique())
    
    for thickness_val in unique_thicknesses:
        df_thick = df_full[df_full['キャビ厚'] == thickness_val]
        grad_settings = CAVITY_THICKNESS_GRADIENT_SETTINGS.get(thickness_val, {'start_color':'lightgray', 'end_color':'black'})
        color_grad = generate_color_gradient(grad_settings['start_color'], grad_settings['end_color'], len(unique_temps))
        
        for i, temp_val in enumerate(unique_temps):
            df_temp = df_thick[df_thick['温度'] == temp_val]
            if df_temp.empty: continue
            temp_label = f"{int(temp_val)}℃"
            marker = TEMP_MARKERS.get(temp_label, 'x')
            color = color_grad[i % len(color_grad)]
            if df_temp[x_col].le(0).any() or df_temp[y_col].le(0).any(): continue
            ax.scatter(df_temp[x_col], df_temp[y_col], marker=marker, color=color, s=SCATTER_MARKER_SIZE)

    ax.set_title(f"[{serial_num}] {graph_title}"); ax.set_xlabel(x_col); ax.set_ylabel(y_col); ax.grid(False)
    if xlims: ax.set_xlim(xlims)
    if ylims: ax.set_ylim(ylims)
    set_manual_log_ticks(ax, xlims, ylims)

    # ★★★【修正】凡例を単一化し、自動配置('best')と最前面表示(zorder)を設定 ★★★
    from matplotlib.lines import Line2D
    dummy_handle = Line2D([0], [0], marker='o', color='none', linestyle='none')
    temp_handles = [plt.Line2D([0], [0], marker=m, color='gray', linestyle='None') for t, m in TEMP_MARKERS.items() if int(t.replace('℃','')) in unique_temps]
    temp_labels = [t for t, m in TEMP_MARKERS.items() if int(t.replace('℃','')) in unique_temps]
    cavity_handles = [plt.Rectangle((0,0), 1, 1, color=generate_color_gradient(s['start_color'], s['end_color'], 3)[1]) for t, s in CAVITY_THICKNESS_GRADIENT_SETTINGS.items() if t in unique_thicknesses]
    cavity_labels = [f"{t}mm" for t in unique_thicknesses]
    
    all_handles = [dummy_handle] + temp_handles + [dummy_handle] + cavity_handles
    all_labels = ['Temperature'] + temp_labels + ['Cavity Thickness (Gradient)'] + cavity_labels

    legend = ax.legend(all_handles, all_labels, loc='best', frameon=False, fontsize='large')
    for text in legend.get_texts():
        if text.get_text() in ['Temperature', 'Cavity Thickness (Gradient)']:
            text.set_weight('bold')
    
    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_final_individual_rev", y_col)
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO(); fig.savefig(img_data, format='png'); plt.close(fig)
    worksheet.set_column(start_col, start_col + graph_col_span - 1, 8)
    worksheet.insert_image(start_row, start_col, '', {'image_data': img_data})
    return graph_col_span

# --- 凡例逆転版・総集編グラフ (両対数スケール) ---
def create_final_loglog_summary_graph_reversed(df_full, workbook, worksheet, xy_cols, start_row, counter, save_folder_path, start_col=1, xlims=None, ylims=None):
    print(f"  総集編・総合比較グラフ(LogLog, 凡例逆転)を作成中...")
    serial_num = counter.get_next()
    graph_title = '全データ 総合比較（凡例逆転）'
    fig, ax = plt.subplots(figsize=FIGSIZE_LOGLOG_SUMMARY)
    ax.set_xscale('log'); ax.set_yscale('log')
    graph_col_span = 15

    if '温度' not in df_full.columns:
        df_full['温度'] = pd.to_numeric(df_full['元ファイル'].str.extract(r'(\d+)℃')[0], errors='coerce')
    unique_thicknesses = sorted(df_full['キャビ厚'].unique())
    unique_temps = sorted(df_full['温度'].unique())
    
    for thickness_val in unique_thicknesses:
        df_thick = df_full[df_full['キャビ厚'] == thickness_val]
        grad_settings = CAVITY_THICKNESS_GRADIENT_SETTINGS.get(thickness_val, {'start_color':'lightgray', 'end_color':'black'})
        color_grad = generate_color_gradient(grad_settings['start_color'], grad_settings['end_color'], len(unique_temps))
        
        for i, temp_val in enumerate(unique_temps):
            df_temp = df_thick[df_thick['温度'] == temp_val]
            if df_temp.empty: continue
            temp_label = f"{int(temp_val)}℃"
            marker = TEMP_MARKERS.get(temp_label, 'x')
            color = color_grad[i % len(color_grad)]
            for x_col, y_col in xy_cols:
                if df_temp[x_col].le(0).any() or df_temp[y_col].le(0).any(): continue
                ax.scatter(df_temp[x_col], df_temp[y_col], marker=marker, color=color, s=SCATTER_MARKER_SIZE)

    ax.set_title(f"[{serial_num}] {graph_title}"); ax.set_xlabel("Shear Rate (1/s)"); ax.set_ylabel("Viscosity (Pa·s)"); ax.grid(False)
    if xlims: ax.set_xlim(xlims)
    if ylims: ax.set_ylim(ylims)
    set_manual_log_ticks(ax, xlims, ylims)

    # ★★★【修正】凡例を単一化し、自動配置('best')と最前面表示(zorder)を設定 ★★★
    from matplotlib.lines import Line2D
    dummy_handle = Line2D([0], [0], marker='o', color='none', linestyle='none')
    temp_handles = [plt.Line2D([0], [0], marker=m, color='gray', linestyle='None') for t, m in TEMP_MARKERS.items() if int(t.replace('℃','')) in unique_temps]
    temp_labels = [t for t, m in TEMP_MARKERS.items() if int(t.replace('℃','')) in unique_temps]
    cavity_handles = [plt.Rectangle((0,0), 1, 1, color=generate_color_gradient(s['start_color'], s['end_color'], 3)[1]) for t, s in CAVITY_THICKNESS_GRADIENT_SETTINGS.items() if t in unique_thicknesses]
    cavity_labels = [f"{t}mm" for t in unique_thicknesses]
    
    all_handles = [dummy_handle] + temp_handles + [dummy_handle] + cavity_handles
    all_labels = ['Temperature'] + temp_labels + ['Cavity Thickness (Gradient)'] + cavity_labels
    
    legend = ax.legend(all_handles, all_labels, loc='best', frameon=False, fontsize='large')
    for text in legend.get_texts():
        if text.get_text() in ['Temperature', 'Cavity Thickness (Gradient)']:
            text.set_weight('bold')
    
    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_final_summary_rev", "all_series_loglog")
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO(); fig.savefig(img_data, format='png'); plt.close(fig)
    worksheet.set_column(start_col, start_col + graph_col_span - 1, 8)
    worksheet.insert_image(start_row, start_col, '', {'image_data': img_data})
    return graph_col_span

print("8種類の総集編グラフ用関数を定義しました。")

In [ ]:
# =====================================================================================
# セル10: Excelレポートとグラフ画像の自動生成（最終版）
# =====================================================================================
if 'file_paths' in locals() and file_paths:
    print("\n--- Excelレポートとグラフ画像の生成を開始します ---")
    output_folder = os.path.dirname(file_paths[0])
    
    for target_key, df in dataframes.items():
        graph_counter = GraphCounter()
        base_name = os.path.splitext(target_key)[0]
        main_graph_folder = os.path.join(output_folder, f"05_graph_images_{base_name}")
        os.makedirs(main_graph_folder, exist_ok=True)
        print(f"\n画像保存先のメインフォルダを作成しました: {main_graph_folder}")
        excel_output_path = os.path.join(output_folder, f"04_graph_output_{base_name}.xlsx")

        with pd.ExcelWriter(excel_output_path, engine='xlsxwriter') as writer:
            workbook = writer.book
            print(f"--- Excelレポートの作成を開始します ---")
            print(f"対象ファイル: {target_key}")
            print(f"出力先Excel: {excel_output_path}")

            unique_legends = []
            for i in range(0, len(df), 10):
                chunk = df.iloc[i:i+10]
                if chunk.empty: continue
                filename = chunk['元ファイル'].iloc[0]
                match = re.search(r'(\d+℃)', filename)
                legend_label = match.group(1) if match else f'Data {i//10 + 1}'
                if legend_label not in unique_legends:
                    unique_legends.append(legend_label)
            colors = ['blue', 'green', 'red', 'cyan', 'magenta', 'yellow', 'black', 'purple', 'orange', 'brown']
            color_map = {legend: colors[i % len(colors)] for i, legend in enumerate(unique_legends)}
            
            x_col_speed = '射出率(cm³/s)'
            sheet_definitions = {
                '射出率と時間差の関係': ['時間差ΔT23(s)', '時間差ΔT34(s)', '時間差ΔT24(s)'],
                '射出率と流速の関係': ['流速V23(mm/s)', '流速V34(mm/s)', '流速V24(mm/s)'],
                '射出率と流量の関係': ['流量Q23(mm³/s)', '流量Q34(mm³/s)', '流量Q24(mm³/s)'],
                '射出率と圧力差の関係': ['圧力差ΔP23(Pa)', '圧力差ΔP34(Pa)', '圧力差ΔP24(Pa)']
            }
            log_sheet_def = {
                'せん断速度と樹脂粘度の関係': [
                    ('せん断速度γ23(1/s)', '樹脂粘度η23(Pa・s)'),
                    ('せん断速度γ34(1/s)', '樹脂粘度η34(Pa・s)'),
                    ('せん断速度γ24(1/s)', '樹脂粘度η24(Pa・s)'),
                ]
            }
            
            all_sheet_defs = {**sheet_definitions, **log_sheet_def}
            
            sheet_folders = {}
            for sheet_name in all_sheet_defs.keys():
                safe_sheet_name = re.sub(r'[\\/:*?"<>|]', '_', sheet_name)
                subfolder_path = os.path.join(main_graph_folder, safe_sheet_name)
                os.makedirs(subfolder_path, exist_ok=True)
                sheet_folders[sheet_name] = subfolder_path
            
            for sheet_name, cols in all_sheet_defs.items():
                worksheet = workbook.add_worksheet(sheet_name)
                current_row = 1
                print(f"\nシート '{sheet_name}' を作成中...")
                current_save_path = sheet_folders[sheet_name]
                
                # --- 30行ごとのグラフ作成ループ ---
                for i in range(0, len(df), 30):
                    df_chunk_30 = df.iloc[i:i+30]
                    if df_chunk_30.empty: continue
                    
                    thickness_val = df_chunk_30['キャビ厚'].iloc[0]
                    
                    worksheet.merge_range(current_row, 0, current_row, 40, f"--- データ範囲: {i+1}行目 - {i+len(df_chunk_30)}行目 ---", workbook.add_format({'bold': True, 'align': 'center', 'bg_color': '#DDEBF7'}))
                    current_row += 2
                    
                    chunk_index = i // 30
                    limits = {}
                    
                    if sheet_name in sheet_definitions:
                        sheet_settings = MANUAL_AXIS_LIMITS.get(sheet_name, {})
                        default_settings = sheet_settings.get('default', {})
                        chunk_specific_settings = sheet_settings.get(chunk_index, {})
                        xlim = chunk_specific_settings.get('xlim', default_settings.get('xlim'))
                        ylim = chunk_specific_settings.get('ylim', default_settings.get('ylim'))
                        xinterval = chunk_specific_settings.get('xinterval', default_settings.get('xinterval'))
                        yinterval = chunk_specific_settings.get('yinterval', default_settings.get('yinterval'))
                        if xlim is None and not df_chunk_30[x_col_speed].empty:
                            xmin = df_chunk_30[x_col_speed].min() * 0.95; xmax = df_chunk_30[x_col_speed].max() * 1.05
                            xlim = (xmin, xmax)
                        if ylim is None and not df_chunk_30[cols].empty:
                            ymin = df_chunk_30[cols].min().min() * 0.95; ymax = df_chunk_30[cols].max().max() * 1.05
                            ylim = (ymin, ymax)
                        limits = {'xlim': xlim, 'ylim': ylim, 'xinterval': xinterval, 'yinterval': yinterval}
                    elif sheet_name in log_sheet_def:
                        sheet_settings = MANUAL_AXIS_LIMITS.get(sheet_name, {})
                        default_settings = sheet_settings.get('default', {})
                        chunk_specific_settings = sheet_settings.get(chunk_index, {})
                        xlim = chunk_specific_settings.get('xlim', default_settings.get('xlim'))
                        ylim = chunk_specific_settings.get('ylim', default_settings.get('ylim'))
                        x_cols_log = [item[0] for item in cols]
                        y_cols_log = [item[1] for item in cols]
                        if xlim is None:
                            x_data_for_lim = df_chunk_30[df_chunk_30[x_cols_log] > 0][x_cols_log]
                            if not x_data_for_lim.empty:
                                xmin_log = x_data_for_lim.min().min(); xmax_log = x_data_for_lim.max().max()
                                xlim = (xmin_log * 0.9, xmax_log * 1.1)
                        if ylim is None:
                            y_data_for_lim = df_chunk_30[df_chunk_30[y_cols_log] > 0][y_cols_log]
                            if not y_data_for_lim.empty:
                                ymin_log = y_data_for_lim.min().min(); ymax_log = y_data_for_lim.max().max()
                                ylim = (ymin_log * 0.9, ymax_log * 1.1)
                        limits = {'xlim': xlim, 'ylim': ylim}

                    if sheet_name in sheet_definitions:
                        y_unit_match = re.search(r'(\(.*\))', cols[0])
                        y_unit = y_unit_match.group(1) if y_unit_match else ''
                        y_basename = sheet_name.split('と')[1].replace('の関係', '')
                        y_label = f"{y_basename} {y_unit}"
                        for y_col in cols:
                            create_graph_set(df_chunk_30, workbook, worksheet, x_col_speed, y_col, current_row, color_map, graph_counter, thickness_val, current_save_path,
                                             xlims=limits.get('xlim'), ylims=limits.get('ylim'), xinterval=limits.get('xinterval'), yinterval=limits.get('yinterval'))
                        current_row += 20
                        create_summary_graph(df_chunk_30, workbook, worksheet, x_col_speed, cols, current_row, color_map, y_label, graph_counter, thickness_val, current_save_path,
                                             xlims=limits.get('xlim'), ylims=limits.get('ylim'), xinterval=limits.get('xinterval'), yinterval=limits.get('yinterval'))
                        current_row += 20
                    elif sheet_name in log_sheet_def:
                        for x_col, y_col in cols:
                            create_loglog_scatter_set(df_chunk_30, workbook, worksheet, x_col, y_col, current_row, color_map, graph_counter, thickness_val, current_save_path,
                                                      xlims=limits.get('xlim'), ylims=limits.get('ylim'))
                        current_row += 20
                        create_loglog_summary_graph(df_chunk_30, workbook, worksheet, cols, current_row, color_map, graph_counter, thickness_val, current_save_path,
                                                   xlims=limits.get('xlim'), ylims=limits.get('ylim'))
                        current_row += 20
                    current_row += 3
                
                # --- ★★★【修正】全データ総集編グラフ (通常凡例) の横並び配置 ---
                worksheet.merge_range(current_row, 0, current_row, 40, "--- 全データ総集編グラフ (通常凡例) ---", workbook.add_format({'bold': True, 'align': 'center', 'bg_color': '#BFBFBF'}))
                current_row += 2
                
                final_limits_config = MANUAL_AXIS_LIMITS.get(sheet_name, {}).get('all_data', {})
                final_graph_col = 1
                
                if sheet_name in sheet_definitions:
                    final_xlim = final_limits_config.get('xlim')
                    if final_xlim is None and not df[x_col_speed].empty:
                        final_xlim = (df[x_col_speed].min() * 0.95, df[x_col_speed].max() * 1.05)
                    final_ylim = final_limits_config.get('ylim')
                    if final_ylim is None and not df[cols].empty:
                        final_ylim = (df[cols].min().min() * 0.95, df[cols].max().max() * 1.05)
                    
                    for y_col in cols:
                        span = create_final_individual_graph(df, workbook, worksheet, x_col_speed, y_col, current_row, color_map, graph_counter, current_save_path,
                                                      start_col=final_graph_col, xlims=final_xlim, ylims=final_ylim, 
                                                      xinterval=final_limits_config.get('xinterval'), yinterval=final_limits_config.get('yinterval'))
                        final_graph_col += span + HORIZONTAL_GRAPH_GAP
                    
                    span = create_final_summary_graph(df, workbook, worksheet, x_col_speed, cols, current_row, color_map, graph_counter, current_save_path,
                                               start_col=final_graph_col, xlims=final_xlim, ylims=final_ylim, 
                                               xinterval=final_limits_config.get('xinterval'), yinterval=final_limits_config.get('yinterval'))
                    current_row += 35
                
                elif sheet_name in log_sheet_def:
                    final_xlim = final_limits_config.get('xlim')
                    if final_xlim is None:
                        x_cols_log = [item[0] for item in cols]
                        x_data = df[df[x_cols_log] > 0][x_cols_log]
                        if not x_data.empty:
                            final_xlim = (x_data.min().min() * 0.9, x_data.max().max() * 1.1)
                    final_ylim = final_limits_config.get('ylim')
                    if final_ylim is None:
                        y_cols_log = [item[1] for item in cols]
                        y_data = df[df[y_cols_log] > 0][y_cols_log]
                        if not y_data.empty:
                            final_ylim = (y_data.min().min() * 0.9, y_data.max().max() * 1.1)

                    for x_col, y_col in cols:
                        span = create_final_loglog_individual_graph(df, workbook, worksheet, x_col, y_col, current_row, color_map, graph_counter, current_save_path,
                                                             start_col=final_graph_col, xlims=final_xlim, ylims=final_ylim)
                        final_graph_col += span + HORIZONTAL_GRAPH_GAP
                        
                    span = create_final_loglog_summary_graph(df, workbook, worksheet, cols, current_row, color_map, graph_counter, current_save_path,
                                                      start_col=final_graph_col, xlims=final_xlim, ylims=final_ylim)
                    current_row += 35

                # --- ★★★【修正】全データ総集編グラフ（凡例逆転版）の横並び配置 ---
                worksheet.merge_range(current_row, 0, current_row, 40, "--- 全データ総集編グラフ (凡例逆転版) ---", workbook.add_format({'bold': True, 'align': 'center', 'bg_color': '#A9D08E'}))
                current_row += 2
                final_graph_col = 1

                if sheet_name in sheet_definitions:
                    for y_col in cols:
                         span = create_final_individual_graph_reversed(df, workbook, worksheet, x_col_speed, y_col, current_row, graph_counter, current_save_path,
                                                                start_col=final_graph_col, xlims=final_xlim, ylims=final_ylim, 
                                                                xinterval=final_limits_config.get('xinterval'), yinterval=final_limits_config.get('yinterval'))
                         final_graph_col += span + HORIZONTAL_GRAPH_GAP

                    span = create_final_summary_graph_reversed(df, workbook, worksheet, x_col_speed, cols, current_row, graph_counter, current_save_path,
                                                        start_col=final_graph_col, xlims=final_xlim, ylims=final_ylim, 
                                                        xinterval=final_limits_config.get('xinterval'), yinterval=final_limits_config.get('yinterval'))
                    current_row += 35

                elif sheet_name in log_sheet_def:
                    for x_col, y_col in cols:
                        span = create_final_loglog_individual_graph_reversed(df, workbook, worksheet, x_col, y_col, current_row, graph_counter, current_save_path,
                                                                      start_col=final_graph_col, xlims=final_xlim, ylims=final_ylim)
                        final_graph_col += span + HORIZONTAL_GRAPH_GAP

                    span = create_final_loglog_summary_graph_reversed(df, workbook, worksheet, cols, current_row, graph_counter, current_save_path,
                                                               start_col=final_graph_col, xlims=final_xlim, ylims=final_ylim)
                    current_row += 35
                
            print(f"\n--- ファイル '{target_key}' の処理が完了しました ---")
else:
    print("エラー: グラフ作成の前に、ステップ1から実行してください。")